In [3]:
%load_ext sql


In [4]:
%sql sqlite:///인구수_데이터.db


Connecting to 'sqlite:///인구수_데이터.db'

## SQL table보는법을 알려줘

In [5]:
%%sql
SELECT name FROM sqlite_schema WHERE type='table' AND name NOT LIKE 'sqlite_%';

Running query in 'sqlite:///인구수_데이터.db'

name
인구수


## 인구수 테이블의 내용을 보고싶어

In [6]:
%%sql
SELECT * FROM 인구수;

Running query in 'sqlite:///인구수_데이터.db'

도시,시점,남자수,여자수
서울특별시,2011.01,5110837,5201998
서울특별시,2011.02,5110818,5203427
서울특별시,2011.03,5107011,5200772
서울특별시,2011.04,5103919,5197910
서울특별시,2011.05,5100399,5195244
서울특별시,2011.06,5095751,5192533
서울특별시,2011.07,5092201,5190271
서울특별시,2011.08,5089493,5188014
서울특별시,2011.09,5086210,5185670
서울특별시,2011.10,5082622,5183552


## 시점의 .을 기준으로 앞은 연도 뒤는 월이라는 칼럼을  만들고 시점을 뺸 모든것이 들어간 새로운 뷰를 만들어줘 뷰 이름은 인구수_전처리야

In [8]:
%%sql
drop 
CREATE VIEW 인구수_전처리 AS
SELECT 
    도시,
    -- '시점'을 '0000.00' 형태의 문자열로 변환 후 앞의 4글자를 추출하여 연도로 분리
    CAST(SUBSTR(PRINTF('%.2f', 시점), 1, 4) AS INTEGER) AS 연도,
    -- 뒤의 2글자를 추출하여 월로 분리
    CAST(SUBSTR(PRINTF('%.2f', 시점), 6, 2) AS INTEGER) AS 월,
    남자수,
    여자수
FROM 인구수;

Running query in 'sqlite:///인구수_데이터.db'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(sqlite3.OperationalError) near "CREATE": syntax error
[SQL: drop
CREATE VIEW 인구수_전처리 AS
SELECT
    도시,

    CAST(SUBSTR(PRINTF('%.2f', 시점), 1, 4) AS INTEGER) AS 연도,

    CAST(SUBSTR(PRINTF('%.2f', 시점), 6, 2) AS INTEGER) AS 월,
    남자수,
    여자수
FROM 인구수;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)



## RuntimeError: (sqlite3.OperationalError) view 인구수_전처리 already exists 이런 에러가 나 sqlite쓰고있어

In [9]:
%%sql
CREATE VIEW IF NOT EXISTS 인구수_전처리 AS
SELECT 
    도시,
    -- '시점'을 '0000.00' 형태의 문자열로 변환 후 앞의 4글자를 추출하여 연도로 분리
    CAST(SUBSTR(PRINTF('%.2f', 시점), 1, 4) AS INTEGER) AS 연도,
    -- 뒤의 2글자를 추출하여 월로 분리
    CAST(SUBSTR(PRINTF('%.2f', 시점), 6, 2) AS INTEGER) AS 월,
    남자수,
    여자수
FROM 인구수;

Running query in 'sqlite:///인구수_데이터.db'

++
||
++
++

## 인구수 전처리의 뷰를 보고 싶어

In [10]:
%%sql
SELECT * FROM 인구수_전처리;

Running query in 'sqlite:///인구수_데이터.db'

도시,연도,월,남자수,여자수
서울특별시,2011,1,5110837,5201998
서울특별시,2011,2,5110818,5203427
서울특별시,2011,3,5107011,5200772
서울특별시,2011,4,5103919,5197910
서울특별시,2011,5,5100399,5195244
서울특별시,2011,6,5095751,5192533
서울특별시,2011,7,5092201,5190271
서울특별시,2011,8,5089493,5188014
서울특별시,2011,9,5086210,5185670
서울특별시,2011,10,5082622,5183552


## 총인구수와 여자 한명에 남자가 몇명있는지 구해줘 

In [18]:
%%sql
SELECT 
    도시,
    연도,
    월,
    (남자수 + 여자수) AS 총인구수,
    ROUND(CAST(남자수 AS REAL) / 여자수, 4) AS 여자1명당_남자수
FROM 인구수_전처리;

Running query in 'sqlite:///인구수_데이터.db'

도시,연도,월,총인구수,여자1명당_남자수
서울특별시,2011,1,10312835,0.9825
서울특별시,2011,2,10314245,0.9822
서울특별시,2011,3,10307783,0.982
서울특별시,2011,4,10301829,0.9819
서울특별시,2011,5,10295643,0.9817
서울특별시,2011,6,10288284,0.9814
서울특별시,2011,7,10282472,0.9811
서울특별시,2011,8,10277507,0.981
서울특별시,2011,9,10271880,0.9808
서울특별시,2011,10,10266174,0.9805


## 도시를 겹치는거 하나없이 보고싶어

In [11]:
%%sql
SELECT DISTINCT 도시 FROM 인구수_전처리;

Running query in 'sqlite:///인구수_데이터.db'

도시
서울특별시
부산광역시
대구광역시
인천광역시
광주광역시
대전광역시
울산광역시
세종특별자치시
경기도
강원특별자치도


## 경기도의 남자 여자수를 보고 싶어

In [12]:
%%sql
SELECT * 
FROM 인구수_전처리 
WHERE 도시 = '경기도';

Running query in 'sqlite:///인구수_데이터.db'

도시,연도,월,남자수,여자수
경기도,2011,1,5949477,5851755
경기도,2011,2,5956364,5858756
경기도,2011,3,5963948,5865368
경기도,2011,4,5969703,5871697
경기도,2011,5,5976420,5877672
경기도,2011,6,5980594,5882295
경기도,2011,7,5986390,5887823
경기도,2011,8,5993230,5894702
경기도,2011,9,5998445,5900145
경기도,2011,10,6002872,5905142


## 최근 연도와 옛날 연도를 구해줘

In [15]:
%%sql
SELECT 
    MAX(연도) AS 최근_연도, 
    MIN(연도) AS 가장_옛날_연도 
FROM 인구수_전처리;

Running query in 'sqlite:///인구수_데이터.db'

최근_연도,가장_옛날_연도
2026,2011


## 2026년의 경기도 남자 여자수를 보고싶어

In [16]:
%%sql
SELECT 
    연도,
    월,
    도시,
    남자수,
    여자수
FROM 인구수_전처리 
WHERE 연도 = 2026 
  AND 도시 = '경기도';

Running query in 'sqlite:///인구수_데이터.db'

연도,월,도시,남자수,여자수
2026,1,경기도,6897544,6839098
2026,2,경기도,6899712,6842360
2026,3,경기도,6901180,6844493
2026,4,경기도,6903574,6847524
2026,5,경기도,6905139,6849987
2026,6,경기도,6908300,6853483


## 2026년의 경기도와 강원특별자치도의 남자와 여자수를 구하고 싶어

In [17]:
%%sql
SELECT 
    연도,
    월,
    도시,
    남자수,
    여자수
FROM 인구수_전처리 
WHERE 연도 = 2026 
  AND 도시 IN ('경기도', '강원특별자치도');

Running query in 'sqlite:///인구수_데이터.db'

연도,월,도시,남자수,여자수
2026,1,경기도,6897544,6839098
2026,2,경기도,6899712,6842360
2026,3,경기도,6901180,6844493
2026,4,경기도,6903574,6847524
2026,5,경기도,6905139,6849987
2026,6,경기도,6908300,6853483
2026,1,강원특별자치도,758094,749540
2026,2,강원특별자치도,757648,749051
2026,3,강원특별자치도,757936,748907
2026,4,강원특별자치도,758166,749033


## 연도별 12월의 총 인구수를 구해줘

In [19]:
%%sql
SELECT 
    연도,
    월,
    SUM(남자수 + 여자수) AS 총인구수
FROM 인구수_전처리
WHERE 월 = 12
GROUP BY 연도;

Running query in 'sqlite:///인구수_데이터.db'

연도,월,총인구수
2011,12,50734284
2012,12,50948272
2013,12,51141463
2014,12,51327916
2015,12,51529338
2016,12,51696216
2017,12,51778544
2018,12,51826059
2019,12,51849861
2020,12,51829023
